In [1]:
import tiktoken
import torch
import torch.nn as nn
import sys
from pathlib import Path
repo_root = Path.cwd().resolve().parent.parent
sys.path.append(str(repo_root))
from gpt_for_text_generation.src.gpt_with_fattn_and_kv_cache import generate_text_simple_cached, GPTModel, GPTModel_with_KV_Cache_and_FlashAttention

In [ ]:
from math import e
import time
GPT_CONFIG_124M = {
        "vocab_size": 50257,     
        "context_length": 1024,  
        "emb_dim": 768,          
        "n_heads": 12,           
        "n_layers": 12,          
        "drop_rate": 0.1,       
        "qkv_bias": False,   
        "kv_window_size":1024
    }


torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model2 = GPTModel_with_KV_Cache_and_FlashAttention(GPT_CONFIG_124M)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
model2.to(device)
model2.eval()

start_context = "Hello, I am"

tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded, device=device).unsqueeze(0)
start1 = time.time()
out = generate_text_simple_cached(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=200
)
end1 = time.time()

start2 = time.time()
out2 = generate_text_simple_cached(
    model=model2,
    idx=encoded_tensor,
    max_new_tokens=200
)
end2 = time.time()
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
decoded_text2 = tokenizer.decode(out2.squeeze(0).tolist())

print("\nOutput:", out)
print("Output length:", len(out[0]))
print("Output text:", decoded_text)
print("\nOutput2:", out2)
print("Output2 length:", len(out2[0]))
print("Output2 text:", decoded_text2)
print(f"Time taken by model1: {end1 - start1:.2f} sec")
print(f"Time taken by model2: {end2 - start2:.2f} sec")

if torch.cuda.is_available():
    max_mem_bytes = torch.cuda.max_memory_allocated()
    max_mem_gb = max_mem_bytes/(1024**3)
    print(f"maximum memory allocated: {max_mem_gb:.2f} GB")




Output: tensor([[15496,    11,   314,   716, 10264, 23139, 20109,  2583, 16896, 29091,
         42730, 34612, 30165, 42315, 37113, 46576, 27420, 41394, 26098, 30499,
         42052, 46812, 36793,  3719, 34462, 47458, 37286, 28395,  6303,  5832,
         17104,  8201,  9121,  4367,  6377, 26242,  1795, 20103, 32119, 31243,
          7753, 43076, 27849,  9485, 27789, 50065, 12377,  7585, 33879, 25779,
         36006, 34196, 31044, 44688, 22294, 33423, 34683,  6993, 46751,  4587,
         20562, 35236, 27006, 44109,  9616, 42497, 44043, 23914, 27898, 21822,
         17641,  1801, 47548, 47458, 13136, 43247, 20540,  5649,  6013, 24622,
          2551, 32338, 12673,  5906, 29303, 27603, 28013, 19284, 43974, 23344,
         11593, 15004, 37917, 34780, 47215, 50024, 11285, 25970, 42730, 20940,
          4647, 10409, 48282, 27561,  8513, 22230, 24152, 30942, 33879,  1555,
          4518, 22939, 30548, 17177, 13562, 22342, 48714, 30845,  4667, 18921,
         14647,  8126,  3100, 38012,  9982,

In [13]:
from gpt_for_text_generation.src.gpt import create_dataloader_v1
from pretraining.src.pretrain import train_model_simple


with open("../../tokenizer/the-verdict.txt", "r") as f:
    text_data = f.read()

train_ratio = 0.9
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

batch_max_length = 256

torch.manual_seed(123)
train_loader = create_dataloader_v1(
    text=train_data, 
    batch_size=2,
    max_length=batch_max_length,
    stride=batch_max_length,
    shuffle=True, 
    drop_last=True,
    num_workers=0
    )

val_loader = create_dataloader_v1(
    text=val_data, 
    batch_size=2,
    max_length=batch_max_length,
    stride=batch_max_length,
    shuffle=True, 
    drop_last=True,
    num_workers=0
    )

In [11]:
# optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.01)
num_epochs = 10

# train_losses, val_losses, tokens_seen = train_model_simple(model, train_loader=train_loader, val_loader=val_loader, optimizer=optimizer, device=device, num_epochs=num_epochs, eval_freq=5, eval_iter=5, start_context="every effort moves you", tokenizer=tokenizer)

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 